<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day09-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 9, Part 2 discussion — Where does the hidden layer's advantage come from?

On the book page, a 64/64 network beat linear regression on California
housing by about a third (test MAE 0.345 vs. 0.534). The page's
explanation: house prices depend on *location* in a way no weighted sum of
latitude and longitude can capture.

**Test that explanation.** Train both models on three feature sets:

1. all 8 features;
2. everything *except* latitude and longitude;
3. *only* latitude and longitude.

**Predict first:** in which setting will the network's advantage over the
linear model be largest? Smallest? Will the network still win at all
without location?

In [1]:
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

ch = fetch_california_housing()
X_all, y = ch.data.astype(np.float32), ch.target.astype(np.float32)
idx = np.arange(len(y))
tr, tmp = train_test_split(idx, test_size=0.30, random_state=0)
va, te = train_test_split(tmp, test_size=0.5, random_state=0)
to_t = lambda a: torch.tensor(a, dtype=torch.float32)
mae = lambda out, t: (out - t).abs().mean().item()

def train(model, Xtr, ytr, Xva, yva, epochs=60, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    best = (float("inf"), None)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); nn.functional.mse_loss(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v = mae(model(Xva), yva)
        if v < best[0]:
            best = (v, {k: w.clone() for k, w in model.state_dict().items()})
    model.load_state_dict(best[1])
    return model

In [2]:
feature_sets = {
    "all 8 features":       list(range(8)),
    "no latitude/longitude": [0, 1, 2, 3, 4, 5],
    "only latitude/longitude": [6, 7],
}
print(f"{'features':26s} {'linear':>8s} {'64/64 net':>10s} {'gap':>7s}   (test MAE, x $100,000)")
for name, cols in feature_sets.items():
    X = X_all[:, cols]
    mu, sd = X[tr].mean(0), X[tr].std(0)
    Xs = (X - mu) / sd
    Xtr, Xva, Xte = to_t(Xs[tr]), to_t(Xs[va]), to_t(Xs[te])
    ytr, yva, yte = [to_t(y[i]).reshape(-1, 1) for i in (tr, va, te)]
    k = len(cols)
    torch.manual_seed(0); lin = train(nn.Linear(k, 1), Xtr, ytr, Xva, yva)
    torch.manual_seed(0); net = train(nn.Sequential(nn.Linear(k, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(),
                                                     nn.Linear(64, 1)), Xtr, ytr, Xva, yva)
    with torch.no_grad():
        a, b = mae(lin(Xte), yte), mae(net(Xte), yte)
    print(f"{name:26s} {a:8.3f} {b:10.3f} {a - b:7.3f}")

features                     linear  64/64 net     gap   (test MAE, x $100,000)


all 8 features                0.534      0.345   0.189


no latitude/longitude         0.580      0.445   0.135


only latitude/longitude       0.763      0.591   0.172


**Discuss:**

1. With *only* latitude and longitude, what can the linear model represent,
   geometrically? What can the network represent that it cannot?
2. Without latitude and longitude the network still wins. Which of the
   remaining features might combine non-additively? (Think about median
   income together with average rooms, or occupancy.)
3. Find a biological analogue: a prediction problem where two inputs matter
   mostly *in combination*, like latitude and longitude. (Hint: Day 6's
   zinc-finger example, or epistasis between mutations.)